# Optimización de Crop4

Se maximiza el SSIM medio sobre las mismas mezclas de entrenamiento y con la misma semilla en cada prueba. La mezcla usa pesos iguales.


In [1]:
%pip install optuna==4.6.0
%pip install matplotlib==3.10.8

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [13]:
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import optuna

project_root = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "project" / "CROP" / "models").is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from project.loader import Loader
from project.CROP.models.crop2 import Crop2


SEED = 1234

DATASET = "fashion"
INTER= 256
LAT=64 
CVAE_NAME = f"int_{INTER}_lat_{LAT}_{DATASET}.keras"
PREDICTOR_NAME = f"CCE_Conv2D_{DATASET}.keras"
N_IMAGES = 1000
N_TRIALS = 100
TARGET_METRIC = "acc_both"  # Maximizar el SSIM medio de las estimaciones.
SEPARATION_ITERATIONS = 30
assert TARGET_METRIC in {"acc_both", "ssim", "recon_bpsnr"}

FIXED_PARAMS = {
    "alpha_1": -2,
    "alpha_2": -22,
    "bias": 0.22,
    "slope": 22,
    "alpha_mix": 0.5,
}

tf.keras.utils.set_random_seed(SEED)
data = Loader.data(dataset=DATASET)
cvae = Loader.cvae(model_name=CVAE_NAME)
predictor = Loader.predictor(model_name=PREDICTOR_NAME)

# Generador independiente de las semillas del CVAE y de Optuna.
rng = np.random.default_rng(SEED)
idx0 = rng.permutation(len(data["x_train"]))[:N_IMAGES]
idx1 = rng.permutation(len(data["x_train_1"]))[:N_IMAGES]
train_batch = (
    data["x_train"][idx0], data["x_train_1"][idx1],
    data["y_train"][idx0], data["y_train_1"][idx1],
)


def scalar_metric(metrics, key):
    value = metrics[key]
    # Las métricas de calidad devuelven (media, desvío).
    if isinstance(value, (tuple, list)):
        value = value[0]
    return float(value)


def objective(trial):
    params = {
        **FIXED_PARAMS,
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
    }
    tf.keras.utils.set_random_seed(SEED)
    model = Crop2(cvae=cvae, predictor=predictor)
    metrics = model.unmix(
        *train_batch,
        params=params,
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        show_image=False,
        show_metrics=False,
    )
    score = scalar_metric(metrics, TARGET_METRIC)
    if not np.isfinite(score):
        raise ValueError(f"Métrica no finita: {TARGET_METRIC}={score}")
    trial.set_user_attr("effective_params", params)
    for key in ("acc_both", "ssim", "recon_bpsnr"):
        trial.set_user_attr(key, scalar_metric(metrics, key))
    return score


Usando fashion como dataset


## Estudio

Se ajusta `delta` entre 0 y 1 en escala lineal. Se ajustan `confidence_scale` y `confidence_floor` entre 0.01 y 1 en escala logarítmica. Cada prueba usa 30 iteraciones de separación y se guarda en SQLite.


In [14]:
from datetime import datetime

output_dir = project_root / "project" / "experiments" / "optim_params" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
study_path = output_dir / f"crop4_{DATASET}_{TARGET_METRIC}_{run_id}.sqlite3"
study = optuna.create_study(
    direction="maximize",
    storage=f"sqlite:///{study_path}",
    study_name=f"Crop4_{DATASET}_{TARGET_METRIC}",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
print("Estudio persistente:", study_path, flush=True)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
best_params = {**FIXED_PARAMS, **study.best_params}
print("Mejor valor:", study.best_value)
print("Parámetros completos:", best_params)


[I 2026-11-24 23:03:11,229] A new study created in RDB with name: Crop4_fashion_acc_both


Estudio persistente: /mnt/c/Users/Santi galan/Desktop/repo_tesis/project/experiments/optim_params/results/crop4_fashion_acc_both_20261124_230311.sqlite3


[I 2026-11-24 23:03:13,531] Trial 0 finished with value: 0.627 and parameters: {'gamma': 0.3830389007577846}. Best is trial 0 with value: 0.627.
[I 2026-11-24 23:03:15,398] Trial 1 finished with value: 0.506 and parameters: {'gamma': 1.2442175420796637}. Best is trial 0 with value: 0.627.
[I 2026-11-24 23:03:17,097] Trial 2 finished with value: 0.592 and parameters: {'gamma': 0.875455478014229}. Best is trial 0 with value: 0.627.
[I 2026-11-24 23:03:18,775] Trial 3 finished with value: 0.424 and parameters: {'gamma': 1.5707171674275384}. Best is trial 0 with value: 0.627.
[I 2026-11-24 23:03:20,456] Trial 4 finished with value: 0.428 and parameters: {'gamma': 1.559951616237607}. Best is trial 0 with value: 0.627.
[I 2026-11-24 23:03:22,092] Trial 5 finished with value: 0.632 and parameters: {'gamma': 0.5451852105652832}. Best is trial 5 with value: 0.632.
[I 2026-11-24 23:03:22,524] Trial 6 finished with value: 0.632 and parameters: {'gamma': 0.5529285102861934}. Best is trial 5 with v

Mejor valor: 0.633
Parámetros completos: {'alpha_1': -2, 'alpha_2': -22, 'bias': 0.22, 'slope': 22, 'alpha_mix': 0.5, 'gamma': 0.5371243392796761}


## Visualización de la optimización

Gráficos con Matplotlib: no requieren Plotly ni instalaciones dentro del notebook.

In [15]:
from optuna.visualization import plot_slice

plot_slice(study, params=list(study.best_params), target_name=TARGET_METRIC)


_____________________


In [1]:
#optim + sepración
import os
import random
import numpy as np
import tensorflow as tf
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

SEED = 1234

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

from project.loader import Loader
from project.CROP.models.crop2 import Crop2

dataset = "fashion"

lat=64
inter=256
data = Loader.data(dataset=dataset)
predictor = Loader.predictor(dataset=dataset)

x_train = data["x_train"]
x_train_1 = data["x_train_1"]
y_train = data["y_train"]
y_train_1 = data["y_train_1"]


c = Crop2(predictor=predictor,lat=lat,inter=inter,dataset=dataset)

def objective(trial):

    gamma = trial.suggest_float("gamma", 0, 2)

    n_images=1000
    try:
        metrics = c.unmix(
            x_train[:n_images],
            x_train_1[:n_images],
            y_train[:n_images],
            y_train_1[:n_images],
            params={"gamma": gamma},                
            iterations=30,
            show_metrics=False
        )
        met = float(metrics["acc_both"])
        return  met

    except Exception as e:
        print(f"Error con gamma={gamma}: {e}")
        return float("inf")


import optuna

optuna.samplers.TPESampler(seed=SEED, multivariate=True)
study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials=100)

# Mostramos los mejores resultados
print("\n📊 Mejores hiperparámetros encontrados:")
print(study.best_trials[0].params)

I0000 00:00:1795570607.123150   20148 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795570607.168794   20148 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795570609.203680   20148 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Usando fashion como dataset


I0000 00:00:1795570610.852887   20148 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9534 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6
/home/galsan/venvs/tensorflow-gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/galsan/venvs/tensorflow-gpu/lib/python3.10/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-11-24 22:36:52,013] A new study created in memory with name: no-name-1dce5f9f-d1fa-4c89-9aff-03acee866ef8
I0000 00:00:1795570612.493200   20148 cuda_dnn.cc:461] Loaded cuDNN version 92801
[I 2026-11-24 22:36:55,601] Trial 0 finished with value: 0.348 and paramete


📊 Mejores hiperparámetros encontrados:
{'gamma': 0.3819171762084015}


In [9]:
%pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 55.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]━━━━ 1/2 [plotly]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from optuna.visualization import plot_slice

plot_slice(study, params=list(study.best_params), target_name="acc_both")
